### Step 1: Read the source and target

In [0]:
incremental_silver_orders = spark.table(
    "workspace.ecommerce_project.incremental_silver_orders"
)

silver_orders = spark.table(
    "workspace.ecommerce_project.silver_orders"
)

### Step 2: Understand the UPSERT candidates

In [0]:
existing_orders_df = (
    incremental_silver_orders
    .join(
        silver_orders,
        on="order_id",
        how="inner"
    )
)

In [0]:
new_orders_df = (
    incremental_silver_orders
    .join(
        silver_orders,
        on="order_id",
        how="left_anti"
    )
)

In [0]:
print("updated_records_count:",existing_orders_df.count())
print("new_orders_count:",new_orders_df.count())

updated_records_count: 39
new_orders_count: 0


In [0]:
1.why new_orders_count is zero beacuse we have already inserted the records into the silver orders so it shows zero 

2.why updated records so 39 as we have loaded all the new data and updated data into silver orders and we are using inner join it display the total count of incremental orders


  File <command-6516391969239046>, line 1
    why new_orders_count is zero beacuse we have already inserted the records into the silver orders so it shows zero
        ^
SyntaxError: invalid syntax


### Step 3: Perform the UPSERT

In [0]:
from delta.tables import DeltaTable

silver_target = DeltaTable.forName(
    spark,
    "workspace.ecommerce_project.silver_orders"
)

In [0]:
(
    silver_target.alias("target")
    .merge(
        incremental_silver_orders.alias("source"),
        "target.order_id=source.order_id"
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

### Step 4: Verify the Silver table

In [0]:
silver_orders_after_upsert = spark.table(
    "workspace.ecommerce_project.silver_orders"
)

print("Silver records after UPSERT:",silver_orders_after_upsert.count()
      )

Silver records after UPSERT: 738


In [0]:
from pyspark.sql import functions as F
silver_orders_after_upsert.filter(
    F.col("order_id").isin(
        ['ORD100208','ORD100958']
    )
).show()

+---------+-----------+-------------+--------------------+--------------+----------+-------------+--------+----------+----------------+------------+------------+----------+----------+-------------+--------------+-------+--------------------+---------------+--------+-------------+-----------+--------------+------------+--------------+-----------+-------------+---------------------+----------------------+----------------------+----------------+-------------------+------------------+--------------------+----------------+----------------+------------------+------------------------+--------------------+--------------------+----------------------+----------------------+--------------------+------------------+-----------------+---------------------+-----------------------+--------------------------+--------------------+--------------------+--------------------+--------------------------+---------------------+------------------------------+---------------------+----------------------+---------